In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS tfl_pipeline")
spark.sql("CREATE SCHEMA IF NOT EXISTS tfl_pipeline.gold")

In [0]:
import sys
from datetime import datetime, timezone

from pyspark.sql import functions as F

sys.path.append("/Workspace/Users/ichibakry7@gmail.com/tfl-tracker/src")

from config import ( 
    SILVER_ARRIVALS,
    SILVER_DISRUPTIONS,
    SILVER_STOP_POINTS,
    GOLD_NETWORK_SNAPSHOT_SUMMARY,
)

def log(msg):
    print(f"[{datetime.now(timezone.utc).isoformat()}] {msg}")



log("Starting gold network snapshot summary transform...")

In [0]:
arrivals = spark.table(SILVER_ARRIVALS)

# NETWORK SNAPSHOT SUMMARY
#
# Grain:
#   one row per observed minute, line, station, and direction
#
# Why this grain:
#   one API pull can span multiple minutes, so keeping the observed minute
#   preserves the shape of the live snapshot instead of flattening it into
#   a single daily row.

arrivals_snapshot = (
    arrivals
    .withColumn(
        "snapshot_observed_at",
        F.date_trunc("minute", F.col("prediction_timestamp"))
    )
    .withColumn("snapshot_date", F.to_date(F.col("prediction_timestamp")))
    .groupBy(
        "snapshot_observed_at",
        "snapshot_date",
        "line_id",
        "line_name",
        "naptan_id",
        "station_name",
        "direction",
    )
    .agg(
        F.count("*").alias("prediction_count"),
        F.countDistinct("vehicle_id").alias("distinct_vehicle_count"),
        F.round(F.avg("time_to_station_sec"), 1).alias("avg_time_to_station_sec"),
        F.round(
            F.percentile_approx("time_to_station_sec", 0.5),
            1
        ).alias("median_time_to_station_sec"),
        F.min("time_to_station_sec").alias("min_time_to_station_sec"),
        F.max("time_to_station_sec").alias("max_time_to_station_sec"),
        F.countDistinct("destination_naptan").alias("distinct_destinations"),
    )
)


In [0]:
stops = spark.table(SILVER_STOP_POINTS)

stops_lookup = (
    stops
    .select(
        "naptan_id",
        "common_name",
        "zone",
        "lat",
        "lon",
        "stop_type",
        "is_active",
        "has_lifts",
        "has_wifi",
        "has_toilets",
        "has_car_park",
        "has_waiting_room",
        "help_points",
    )
    .dropDuplicates(["naptan_id"])
)

In [0]:
# Disruptions only have ingested_at in Silver, so this join assumes the
# arrivals and disruptions ingested on the same calendar date belong to the
# same live snapshot. If a pull crosses midnight, a disruption could miss
# the join and should be called out as a limitation.

disruptions = spark.table(SILVER_DISRUPTIONS)


disruptions_by_line_date = (
    disruptions
    .withColumn("snapshot_date", F.to_date(F.col("ingested_at")))
    .groupBy("snapshot_date", "line_id")
    .agg(
        F.count("*").alias("active_disruption_count"),
        (F.count("*") > 0).alias("has_active_disruption"),
        F.collect_set("category").alias("disruption_categories"),
        F.collect_set("disruption_type").alias("disruption_types"),
        F.collect_set("description").alias("disruption_descriptions"),
    )
)





In [0]:
gold_network_snapshot_summary = (
    arrivals_snapshot
    .join(stops_lookup, on="naptan_id", how="left")
    .join(disruptions_by_line_date, on=["snapshot_date", "line_id"], how="left")
    .withColumn(
        "has_active_disruption",
        F.coalesce(F.col("has_active_disruption"), F.lit(False))
    )
    .withColumn(
        "active_disruption_count",
        F.coalesce(F.col("active_disruption_count"), F.lit(0))
    )
    .select(
        "snapshot_observed_at",
        "snapshot_date",
        "line_id",
        "line_name",
        "naptan_id",
        "station_name",
        "common_name",
        "direction",
        "zone",
        "lat",
        "lon",
        "stop_type",
        "is_active",
        "has_lifts",
        "has_wifi",
        "has_toilets",
        "has_car_park",
        "has_waiting_room",
        "help_points",
        "prediction_count",
        "distinct_vehicle_count",
        "avg_time_to_station_sec",
        "median_time_to_station_sec",
        "min_time_to_station_sec",
        "max_time_to_station_sec",
        "distinct_destinations",
        "has_active_disruption",
        "active_disruption_count",
        "disruption_categories",
        "disruption_types",
        "disruption_descriptions",
    )
)


In [0]:
(gold_network_snapshot_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_NETWORK_SNAPSHOT_SUMMARY))

log(f"gold_network_snapshot_summary: wrote {gold_network_snapshot_summary.count()} rows")
log("Gold transform complete.")